### 2.1 理论计算题

**已知：**
- 输入：$3 \times 32 \times 32$（通道×高×宽）
- 卷积核：16个，每个 $3 \times 5 \times 5$
- Padding = 2，Stride = 2

**1. 输出特征图尺寸**

输出尺寸公式：
$$H_{out} = \left\lfloor \frac{H_{in} + 2p - k}{s} \right\rfloor + 1$$

计算：
$$H_{out} = \left\lfloor \frac{32 + 2 \times 2 - 5}{2} \right\rfloor + 1 = \left\lfloor \frac{31}{2} \right\rfloor + 1 = 15 + 1 = 16$$

输出：**16 × 16 × 16**（通道数×高×宽）

**2. 单个输出像素的乘法次数**

单个输出通道的一个像素，对应卷积核 $3 \times 5 \times 5$ 与输入对应区域做点乘：

$$3 \times 5 \times 5 = 75 \text{ 次乘法}$$

> 注意：如果考虑偏置项不计入乘法，此处为纯乘法次数。

---

### 2.2 编程题 - 二维最大池化前向传播


In [1]:
import numpy as np

def max_pool2d_manual(input, kernel_size, stride=1, padding=0):
    """
    input: (C, H, W) 或 (H, W)
    kernel_size: int
    stride: int
    padding: int
    """
    if input.ndim == 2:
        input = input[np.newaxis, ...]  # (1, H, W)
    C, H, W = input.shape
    K = kernel_size

    # 填充
    input_padded = np.pad(input, ((0, 0), (padding, padding), (padding, padding)), mode='constant')

    H_out = (H + 2*padding - K) // stride + 1
    W_out = (W + 2*padding - K) // stride + 1

    output = np.zeros((C, H_out, W_out))

    for c in range(C):
        for i in range(H_out):
            for j in range(W_out):
                h_start = i * stride
                h_end = h_start + K
                w_start = j * stride
                w_end = w_start + K
                region = input_padded[c, h_start:h_end, w_start:w_end]
                output[c, i, j] = np.max(region)

    return output

# 示例
if __name__ == "__main__":
    x = np.random.randn(3, 32, 32)
    out = max_pool2d_manual(x, kernel_size=3, stride=2, padding=1)
    print(out.shape)  # 期望 (3, 16, 16)

(3, 16, 16)


### 3.1 理论计算题

**1. 单个 $5 \times 5$ 卷积层的参数量**

参数量 = 输入通道数 × 输出通道数 × 卷积核高 × 卷积核宽

$$Params = C \times C \times 5 \times 5 = 25C^2$$

**2. 两个串联 $3 \times 3$ 卷积层的总参数量**

每层参数量：$C \times C \times 3 \times 3 = 9C^2$

两层总计：$9C^2 + 9C^2 = 18C^2$

> 结论：两个 $3 \times 3$ 卷积核（18C²）比一个 $5 \times 5$ 卷积核（25C²）参数量更少，同时具有更大的有效感受野（5×5）。

---

### 3.2 编程题 - NiN块

In [2]:
import torch
import torch.nn as nn

class NiNBlock(nn.Module):
    """
    NiN块：普通卷积 + 两个1x1卷积，每层后接ReLU
    """
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding):
        super(NiNBlock, self).__init__()
        self.block = nn.Sequential(
            # 普通卷积层
            nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding),
            nn.ReLU(inplace=True),
            # 1x1卷积层（充当MLP）
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU(inplace=True),
            # 第二个1x1卷积层
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        return self.block(x)

# 测试代码
if __name__ == "__main__":
    block = NiNBlock(in_channels=3, out_channels=96, kernel_size=5, stride=1, padding=2)
    x = torch.randn(1, 3, 224, 224)
    out = block(x)
    print(f"NiN块输出形状: {out.shape}")

NiN块输出形状: torch.Size([1, 96, 224, 224])


### 4.1 理论计算题

已知：$x = [2, 4, 6, 8]$，$\gamma = 2$，$\beta = 1$，$\epsilon = 0$

**步骤1：计算均值和方差**

$$\mu = \frac{2+4+6+8}{4} = 5$$

$$\sigma^2 = \frac{(2-5)^2 + (4-5)^2 + (6-5)^2 + (8-5)^2}{4} = \frac{9+1+1+9}{4} = \frac{20}{4} = 5$$

**步骤2：归一化**

$$\hat{x}_i = \frac{x_i - \mu}{\sqrt{\sigma^2 + \epsilon}}$$

$$\hat{x}_1 = \frac{2-5}{\sqrt{5}} = \frac{-3}{\sqrt{5}} = -\frac{3\sqrt{5}}{5}$$
$$\hat{x}_2 = \frac{4-5}{\sqrt{5}} = \frac{-1}{\sqrt{5}} = -\frac{\sqrt{5}}{5}$$
$$\hat{x}_3 = \frac{6-5}{\sqrt{5}} = \frac{1}{\sqrt{5}} = \frac{\sqrt{5}}{5}$$
$$\hat{x}_4 = \frac{8-5}{\sqrt{5}} = \frac{3}{\sqrt{5}} = \frac{3\sqrt{5}}{5}$$

**步骤3：缩放和平移**

$$y_i = \gamma \hat{x}_i + \beta = 2\hat{x}_i + 1$$

$$y_1 = 2 \times (-\frac{3\sqrt{5}}{5}) + 1 = -\frac{6\sqrt{5}}{5} + 1$$
$$y_2 = 2 \times (-\frac{\sqrt{5}}{5}) + 1 = -\frac{2\sqrt{5}}{5} + 1$$
$$y_3 = 2 \times (\frac{\sqrt{5}}{5}) + 1 = \frac{2\sqrt{5}}{5} + 1$$
$$y_4 = 2 \times (\frac{3\sqrt{5}}{5}) + 1 = \frac{6\sqrt{5}}{5} + 1$$

**数值近似：** $\sqrt{5} \approx 2.236$

$$y_1 \approx -2.683 + 1 = -1.683$$
$$y_2 \approx -0.894 + 1 = 0.106$$
$$y_3 \approx 0.894 + 1 = 1.894$$
$$y_4 \approx 2.683 + 1 = 3.683$$

---

### 4.2 编程题 - ResNet残差块

In [3]:
import torch
import torch.nn as nn

class Residual(nn.Module):
    """
    残差块：两个3x3卷积层 + 批量归一化 + 可选的1x1跳跃连接
    """
    def __init__(self, in_channels, out_channels, use_1x1conv=False, stride=1):
        super(Residual, self).__init__()
        
        # 第一个卷积层
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, 
                               stride=stride, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        
        # 第二个卷积层
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                               stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # 可选的1x1卷积（调整输入形状）
        if use_1x1conv:
            self.conv3 = nn.Conv2d(in_channels, out_channels, kernel_size=1,
                                   stride=stride)
        else:
            self.conv3 = None
        
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, x):
        # 残差路径
        y = self.relu(self.bn1(self.conv1(x)))
        y = self.bn2(self.conv2(y))
        
        # 恒等映射路径
        if self.conv3 is not None:
            x = self.conv3(x)
        
        # 相加并激活
        out = self.relu(y + x)
        return out

# 测试代码
if __name__ == "__main__":
    # 测试1：通道数相同，无需1x1卷积
    block1 = Residual(64, 64, use_1x1conv=False)
    x1 = torch.randn(1, 64, 56, 56)
    out1 = block1(x1)
    print(f"相同通道数残差块输出: {out1.shape}")
    
    # 测试2：通道数不同，需要1x1卷积
    block2 = Residual(64, 128, use_1x1conv=True, stride=2)
    x2 = torch.randn(1, 64, 56, 56)
    out2 = block2(x2)
    print(f"不同通道数残差块输出: {out2.shape}")

相同通道数残差块输出: torch.Size([1, 64, 56, 56])
不同通道数残差块输出: torch.Size([1, 128, 28, 28])


### 5.1 理论计算题

**1. 为什么底层设置小学习率，顶层设置大学习率？**

- **底层特征提取层**：学习通用的低级特征（边缘、纹理等），这些特征在不同任务间具有迁移性。用较小学习率可以保留预训练学到的通用知识，避免被新数据集"破坏"。

- **顶层输出层**：需要适配新数据集的类别分布，且通常是随机初始化的。用较大学习率可以让它快速学习新任务的特定模式。

**2. 目标数据集很小且与源数据集相似时的策略**

- 冻结大部分底层特征提取层（设置学习率为0）
- 只微调最后几层（通常2-3层）和新初始化的输出层
- 使用较小的学习率（如1e-5到1e-4）
- 使用更强的正则化（Dropout、权重衰减、数据增广）
- 使用较小的批量大小

---

### 5.2 编程题 - 图像增广管道

In [4]:
import torchvision.transforms as transforms
from torchvision.transforms import RandomResizedCrop, RandomHorizontalFlip, ColorJitter, ToTensor

# 创建增广管道
augmentation_pipeline = transforms.Compose([
    # 1. 随机裁剪并缩放到224x224
    transforms.RandomResizedCrop(224, scale=(0.08, 1.0)),
    
    # 2. 50%概率水平翻转
    transforms.RandomHorizontalFlip(p=0.5),
    
    # 3. 随机改变亮度、对比度、饱和度（变化范围0.5）
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
    
    # 4. 转换为张量
    transforms.ToTensor()
])

# 测试代码（需要PIL图像）
if __name__ == "__main__":
    from PIL import Image
    import requests
    from io import BytesIO
    
    # 下载示例图片
    url = "https://picsum.photos/300/200"
    response = requests.get(url)
    img = Image.open(BytesIO(response.content)).convert('RGB')
    
    print(f"原始图像大小: {img.size}")
    
    # 应用增广
    augmented_img = augmentation_pipeline(img)
    print(f"增广后张量形状: {augmented_img.shape}")
    
    # 演示多次增广的不同效果
    print("\n多次增广示例:")
    for i in range(3):
        aug_result = augmentation_pipeline(img)
        print(f"第{i+1}次增广形状: {aug_result.shape}")

原始图像大小: (300, 200)
增广后张量形状: torch.Size([3, 224, 224])

多次增广示例:
第1次增广形状: torch.Size([3, 224, 224])
第2次增广形状: torch.Size([3, 224, 224])
第3次增广形状: torch.Size([3, 224, 224])


### 6.1 理论计算题

**已知：**
- 真实框 A = [10, 10, 50, 50]（左上角x, 左上角y, 右下角x, 右下角y）
- 预测框 B = [30, 30, 70, 70]

**步骤1：计算交集区域**

交集左上角：$x_{min} = \max(10, 30) = 30$，$y_{min} = \max(10, 30) = 30$

交集右下角：$x_{max} = \min(50, 70) = 50$，$y_{max} = \min(50, 70) = 50$

交集宽度：$w_{intersection} = 50 - 30 = 20$

交集高度：$h_{intersection} = 50 - 30 = 20$

**交集面积**：$Area_{intersection} = 20 \times 20 = 400$

**步骤2：计算并集区域**

A的面积：$Area_A = (50-10) \times (50-10) = 40 \times 40 = 1600$

B的面积：$Area_B = (70-30) \times (70-30) = 40 \times 40 = 1600$

并集面积 = $Area_A + Area_B - Area_{intersection} = 1600 + 1600 - 400 = 2800$

**步骤3：计算IoU**

$$IoU = \frac{Area_{intersection}}{Area_{union}} = \frac{400}{2800} = \frac{1}{7} \approx 0.142857$$

---

### 6.2 编程题 - 标签平滑交叉熵损失

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class LabelSmoothingCrossEntropy(nn.Module):
    """
    标签平滑交叉熵损失
    """
    def __init__(self, epsilon=0.1, reduction='mean'):
        super(LabelSmoothingCrossEntropy, self).__init__()
        self.epsilon = epsilon
        self.reduction = reduction
    
    def forward(self, logits, targets):
        """
        参数:
            logits: 模型输出，形状 (batch_size, num_classes)
            targets: 真实标签，形状 (batch_size,)
        """
        num_classes = logits.size(-1)
        batch_size = logits.size(0)
        
        # 计算log_softmax
        log_probs = F.log_softmax(logits, dim=-1)
        
        if self.reduction == 'none':
            # 不使用reduction的情况
            # 创建平滑后的标签分布
            smooth_targets = torch.full_like(log_probs, self.epsilon / (num_classes - 1))
            smooth_targets.scatter_(1, targets.unsqueeze(1), 1 - self.epsilon)
            loss = -torch.sum(smooth_targets * log_probs, dim=-1)
        else:
            # 使用one_hot + 平滑
            with torch.no_grad():
                # 真实类别的目标概率: 1 - epsilon
                # 其他类别的目标概率: epsilon / (num_classes - 1)
                smooth_targets = torch.zeros_like(log_probs)
                smooth_targets.fill_(self.epsilon / (num_classes - 1))
                smooth_targets.scatter_(1, targets.unsqueeze(1), 1 - self.epsilon)
            
            loss = -torch.sum(smooth_targets * log_probs, dim=-1)
            
            if self.reduction == 'mean':
                loss = loss.mean()
            elif self.reduction == 'sum':
                loss = loss.sum()
        
        return loss

# 测试代码
if __name__ == "__main__":
    # 设置参数
    num_classes = 10
    batch_size = 4
    epsilon = 0.1
    
    # 创建随机logits和标签
    logits = torch.randn(batch_size, num_classes)
    targets = torch.tensor([2, 5, 1, 7])
    
    # 使用标准交叉熵作为对比
    ce_loss = nn.CrossEntropyLoss(reduction='none')
    standard_loss = ce_loss(logits, targets)
    
    # 使用标签平滑损失
    ls_loss = LabelSmoothingCrossEntropy(epsilon=epsilon, reduction='none')
    smooth_loss = ls_loss(logits, targets)
    
    print(f"标准交叉熵损失: {standard_loss}")
    print(f"标签平滑损失(ε={epsilon}): {smooth_loss}")
    print(f"\n损失对比:")
    for i in range(batch_size):
        print(f"  样本{i+1}: 标准={standard_loss[i]:.4f}, 平滑={smooth_loss[i]:.4f}")
    
    # 验证平滑后损失是否变小
    print(f"\n平均损失 - 标准: {standard_loss.mean():.4f}, 平滑: {smooth_loss.mean():.4f}")

标准交叉熵损失: tensor([2.1265, 2.2258, 3.2870, 1.5847])
标签平滑损失(ε=0.1): tensor([2.1753, 2.2502, 3.2241, 1.6987])

损失对比:
  样本1: 标准=2.1265, 平滑=2.1753
  样本2: 标准=2.2258, 平滑=2.2502
  样本3: 标准=3.2870, 平滑=3.2241
  样本4: 标准=1.5847, 平滑=1.6987

平均损失 - 标准: 2.3060, 平滑: 2.3371
